# EDA: Hủy phòng (Epic E3)

Mục tiêu:
1. Xác nhận số liệu trong `docs/BUSINESS_PROBLEM.md`: tỉ lệ hủy chung **37,0%**, City Hotel **41,7%**, Resort Hotel **27,8%**.
2. Xem tỉ lệ hủy theo khách sạn, lead time, kênh, đặt cọc, quốc gia, mùa.
3. Chỉ ra cột dễ gây **data leakage**, làm cơ sở cho `ml/cancellation/features.py`.

Dữ liệu: `database/ml/20_cancellation_eda.sql` (silver, 119.390 booking, kèm cờ `in_gold`). Tỉ lệ hủy luôn tính lại từ tổng: `số booking hủy / số booking` (quy tắc 8).

Chạy: mở notebook từ thư mục gốc repo hoặc `ml/notebooks/`, DB đã có silver + gold. **Clear output trước khi commit.**

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'CLAUDE.md').exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from etl.config import DATABASE_DIR
from etl.db import connect

pd.set_option('display.float_format', '{:,.3f}'.format)

sql = (DATABASE_DIR / 'ml' / '20_cancellation_eda.sql').read_text(encoding='utf-8')
with connect() as conn, conn.cursor() as cur:
    cur.execute(sql)
    df = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
df['arrival_date'] = pd.to_datetime(df['arrival_date'])
df.shape

In [ ]:
def cancel_rate(data: pd.DataFrame, by, min_bookings: int = 0) -> pd.DataFrame:
    """Số booking, số hủy, tỉ lệ hủy (= tổng hủy / tổng booking) theo nhóm."""
    out = data.groupby(by, dropna=False, observed=True).agg(
        bookings=('is_canceled', 'size'), canceled=('is_canceled', 'sum'))
    out['cancel_rate'] = out['canceled'] / out['bookings']
    out['share'] = out['bookings'] / out['bookings'].sum()
    return out[out['bookings'] >= min_bookings]


def bar(rates: pd.DataFrame, title: str):
    """Biểu đồ cột 1 chuỗi: tỉ lệ hủy theo nhóm (1 màu, lưới mờ)."""
    ax = rates['cancel_rate'].plot.bar(color='#2a78d6', width=0.7, figsize=(9, 3.5))
    ax.set_title(title, loc='left')
    ax.set_ylabel('Tỉ lệ hủy')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    plt.xticks(rotation=0)
    plt.show()

## 1. Tỉ lệ hủy chung và theo khách sạn (đối chiếu BUSINESS_PROBLEM.md)

In [ ]:
overall = df['is_canceled'].sum() / len(df)
print(f'Silver: {len(df):,} booking, tỉ lệ hủy {overall:.1%}')
gold = df[df['in_gold']]
print(f'Gold (sau khi lọc booking 0 khách, ADR âm): {len(gold):,} booking, '
      f'tỉ lệ hủy {gold.is_canceled.sum() / len(gold):.1%}')
cancel_rate(df, 'hotel_name')

Kết quả khớp tài liệu: **37,0%** chung, City **41,7%**, Resort **27,8%**. Gold bỏ 181 booking (0 khách hoặc ADR âm), tỉ lệ hủy gần như không đổi (37,1%). Mô hình train và chấm điểm trên tập gold.

## 2. Lead time (số ngày đặt trước)

In [ ]:
bins = [-1, 7, 30, 90, 180, 365, 10_000]
labels = ['0-7', '8-30', '31-90', '91-180', '181-365', '>365']
df['lead_time_band'] = pd.cut(df['lead_time'], bins=bins, labels=labels)
lt = cancel_rate(df, 'lead_time_band')
bar(lt, 'Tỉ lệ hủy theo lead time (ngày)')
lt

Lead time càng dài càng dễ hủy: 0-7 ngày chỉ ~10%, 181-365 ngày ~56%, trên 1 năm ~68%. Đây là feature mạnh và **hợp lệ** (biết ngay lúc đặt).

## 3. Kênh bán (market segment, distribution channel)

In [ ]:
cancel_rate(df, 'market_segment').sort_values('cancel_rate', ascending=False)

In [ ]:
cancel_rate(df, ['hotel_name', 'distribution_channel'], min_bookings=100)

Groups và Online TA hủy nhiều nhất; Direct, Corporate thấp. Kênh TA/TO chiếm phần lớn booking nên quyết định chính sách cọc theo kênh có tác động lớn.

## 4. Đặt cọc (deposit type)

In [ ]:
cancel_rate(df, ['deposit_type'])

In [ ]:
cancel_rate(df, ['deposit_type', 'market_segment'], min_bookings=200)

Nghịch lý: booking **Non Refund** (cọc không hoàn) hủy ~99%. Theo bài báo gốc, đây chủ yếu là booking đoàn qua đại lý (Groups / Offline TA/TO) giữ chỗ số lượng lớn rồi hủy. Thông tin cọc có từ lúc đặt nên **hợp lệ**, nhưng mô hình sẽ học 'Non Refund ⇒ hủy', không phải quan hệ nhân quả: không được diễn giải là 'đòi cọc làm khách hủy'.

## 5. Quốc gia

In [ ]:
top = cancel_rate(df, 'country_code').sort_values('bookings', ascending=False).head(15)
top

In [ ]:
df['is_prt'] = df['country_code'].eq('PRT')
cancel_rate(df, ['is_prt', 'deposit_type'], min_bookings=100)

Bồ Đào Nha (PRT) chiếm ~41% booking và hủy ~57%, trong khi các nước khác ~24%. 97% booking Non Refund là PRT; nhưng chỉ xét No Deposit thì PRT vẫn hủy 39% so với 23%. ⚠ Nghi vấn: quốc tịch thường chỉ được xác nhận lúc check-in, booking hủy có thể giữ giá trị mặc định (PRT). Nếu đúng vậy thì `country_code` mang một phần kết quả. Không đủ bằng chứng để loại bỏ, nên `train.py` chạy thêm model **bỏ quốc gia** để đo mức phụ thuộc (xem `ml/README.md`).

## 6. Mùa vụ và năm đến

In [ ]:
m = cancel_rate(df, 'arrival_month')
bar(m, 'Tỉ lệ hủy theo tháng đến (gộp các năm)')
m

In [ ]:
cancel_rate(df, ['arrival_year', 'hotel_name'])

In [ ]:
pd.crosstab(df['arrival_month'], df['arrival_year'], values=df['is_canceled'], aggfunc='mean').round(3)

Tỉ lệ hủy cao hơn từ tháng 4 đến tháng 10 (~38-42%) và thấp hơn từ tháng 11 đến tháng 3 (~30-35%). Nhưng cùng một tháng, tỉ lệ hủy **khác nhiều giữa các năm** (vd tháng 11: 21% năm 2015, 37% năm 2016; mỗi tháng chỉ có đúng 2 năm dữ liệu): đó là lý do mô hình không dùng trực tiếp `arrival_month`/`arrival_week` (dễ học thuộc từng năm), chỉ dùng `season` và tháng đặt phòng. Chia train/test theo thời gian: train < 2017-01-01, test = 2017.

## 7. Cột dễ gây data leakage

In [ ]:
# (a) reservation_status chính là nhãn
pd.crosstab(df['reservation_status'], df['is_canceled'])

In [ ]:
# (a) reservation_status_date: với booking hủy là NGÀY HỦY (trước ngày đến), với booking không hủy là ngày trả phòng
gap = (df['arrival_date'] - pd.to_datetime(df['reservation_status_date'])).dt.days
gap.groupby(df['is_canceled']).describe()

In [ ]:
# (b) assigned_room_type khác reserved_room_type: gần như chỉ xảy ra với khách đã đến
df['is_room_changed'] = df['reserved_room_type'].ne(df['assigned_room_type'])
cancel_rate(df, 'is_room_changed')

In [ ]:
# (b) booking_changes: số lần sửa tích lũy tới lúc check-in/hủy
df['has_changes'] = df['booking_changes'].gt(0)
cancel_rate(df, 'has_changes')

In [ ]:
# (b) days_in_waiting_list
df['waited'] = df['days_in_waiting_list'].gt(0)
cancel_rate(df, 'waited')

In [ ]:
# (b) required_car_parking_spaces: có chỗ đỗ xe thì KHÔNG booking nào hủy
df['needs_parking'] = df['required_car_parking_spaces'].gt(0)
cancel_rate(df, 'needs_parking')

## 8. Kết luận cho feature set

| Cột | Kết luận | Lý do |
|---|---|---|
| `reservation_status`, `reservation_status_date` | ⛔ Bỏ | Chính là kết quả / ngày hủy |
| Cột suy ra từ kết quả trong `gold.fact_booking` (`status_date_key`, `room_nights_sold`, `realized_revenue`, `lost_revenue`, `deposit_retained`, `net_lost_revenue`, `cancel_lead_days`) | ⛔ Bỏ | Tính từ `is_canceled` |
| `assigned_room_type` (và `is_room_changed`) | ⛔ Bỏ | Đổi phòng xảy ra lúc check-in: đổi phòng thì hủy ~5% |
| `booking_changes` | ⛔ Bỏ | Tích lũy sau lúc đặt; booking có sửa hủy ít hơn hẳn |
| `days_in_waiting_list` | ⛔ Bỏ | Chốt sau ngày nhập booking, không chắc có lúc chấm điểm (có chờ thì hủy 64%) |
| `required_car_parking_spaces` | ⛔ Bỏ | Có chỗ đỗ xe ⇒ hủy 0% trên 7.409 booking: nhiều khả năng ghi lúc check-in |
| `country_code` | ⚠ Giữ, có đối chứng | Nghi PRT là giá trị mặc định cho booking hủy |
| `deposit_type` | ✔ Giữ | Biết lúc đặt; quan hệ tương quan, không nhân quả |
| `lead_time`, kênh, đại lý, khách hàng, giá, yêu cầu đặc biệt, mùa | ✔ Giữ | Biết lúc đặt |
